# Evaluacija RAG sistema

Ovaj notebook koristi isti setup (embeddinzi, FAISS baza, kvantizovani LLM, `rag_chain`) kao i glavni notebook, i evaluira ga u dva dela:

1. **Evaluacija Retriever-a (RAG)** – Precision, Recall, Hit Rate, MRR
2. **Evaluacija generisanih odgovora** – ROUGE-L, BLEU, BERTScore, Cosine similarity

Napomena: pošto nema OpenAI ključa, **nema LLM-as-judge** evaluacije – koriste se isključivo besplatne, lokalne metrike.

In [1]:
!pip install -q langchain langchain-classic langchain-community langchain-huggingface \
    faiss-cpu transformers torch accelerate sentence-transformers
!pip install -q bitsandbytes
!pip install -q rouge-score nltk bert-score pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 9.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.6 MB/s eta 0:00:00


In [2]:
import os
import time
import torch
import warnings
import logging
import numpy as np
import pandas as pd

from google.colab import drive

from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import PromptTemplate

# biblioteke za evaluaciju
from rouge_score import rouge_scorer
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bert_score_fn
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

/tmp/ipykernel_538/4163300376.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


True

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
warnings.filterwarnings("ignore")

# POTPUNO GASI BITSANDBYTES I AUTOGRAD WARNINGE
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)

# Ugasi Hugging Face poruke
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 160)

In [5]:
db_path = "/content/drive/MyDrive/NLP_proyect/vector_database"
model_id = "/content/drive/MyDrive/NLP_proyect/merged_model"
K = 5  # broj dokumenata koje retriever vraca po upitu

In [6]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2")
vector_db = FAISS.load_local(db_path, embedding_model, allow_dangerous_deserialization=True)
retriever = vector_db.as_retriever(search_kwargs={"k": K})

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [8]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.15,
    do_sample=False,
    return_full_text=False,
    repetition_penalty=1.05,
    no_repeat_ngram_size=4,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id
)
llm = HuggingFacePipeline(pipeline=pipe)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'eos_token_id', 'no_repeat_ngram_size', 'do_sample', 'repetition_penalty', 'pad_token_id', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [9]:
contextualize_q_system_prompt = """Ukoliko postoji istorija razgovora i korisnik postavi pitanje koje se nadovezuje,
reformuliši to pitanje tako da bude samostalno i razumljivo bez istorije razgovora.
NEMOJ odgovarati na pitanje, samo ga reformuliši ako je potrebno, u suprotnom ga vrati identičnog."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_q_prompt)

system_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
1. Ti si stručni medicinski asistent i komuniciraš isključivo na srpskom jeziku.
2. Ako u kontekstu nema informacija potrebnih za odgovor, nemoj izmišljati nego jasno reci: "Nažalost, nemam te informacije u svojoj bazi podataka, najbolje se savetuj sa lekarom ili farmaceutom. Informacije sa interneta mogu biti lažne i štetne."
3. Piši svakodnevnim i lako shvatljivim jezikom.
4. Ukoliko je pitanje na latinici, odgovori na latinici. Ukoliko je pitanje na ćirilici, odgovori na ćirilici.
5. Na kraju svakog odgovora OBAVEZNO dodaj: "Pre upotrebe svakog medikamenta, obavezno se posavetuje sa lekarom ili farmaceutom. Dobijeni odgovor nije zamena za stručno mišljenje, već pomoćna alatka."

Pruženi medicinski kontekst:
{context}<|eot_id|><|start_header_id|>user<|end_header_id|>
{input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

qa_prompt = PromptTemplate(
    template=system_prompt,
    input_variables=["context", "input"]
)

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [10]:
def generisi_odgovor(pitanje):
    """Pokrece ceo RAG lanac za JEDNO, izolovano pitanje (bez istorije razgovora -
    pogodno za evaluaciju, da odgovor zavisi samo od pitanja i baze znanja).
    Vraca (odgovor, lista_povucenih_dokumenata)."""
    t0 = time.time()
    rezultat = rag_chain.invoke({
        "input": pitanje,
        "chat_history": []
    })
    trajanje = time.time() - t0
    odgovor = rezultat["answer"]
    povuceni_dokumenti = rezultat.get("context", [])
    print(f"Q: {pitanje}\nA: {odgovor}\n[{trajanje:.1f}s]\n")
    return odgovor, povuceni_dokumenti

---
# Deo 1: Evaluacija RAG (retrieval) komponente

Ovde se evaluira **samo retriever** – da li vraca prave dokumente za dato pitanje, nezavisno od toga sta LLM kasnije generise.

Za svaki dokument iz `vector_db` identifikator je `metadata["izvor_fajl"]`. Za svako test-pitanje potrebno je definisati skup **relevantnih** `izvor_fajl` vrednosti (ground truth) sa kojima se poredi ono sto retriever stvarno vrati.


Formule (za top-K rezultata jednog upita, K = 5):

- **Precision@K** = (broj relevantnih dokumenata medju vracenih K) / K
- **Recall@K** = (broj relevantnih dokumenata medju vracenih K) / (ukupan broj relevantnih dokumenata za to pitanje)
- **Hit Rate@K** = 1 ako je bar jedan relevantan dokument medju vracenih K, inace 0
- **RR (Reciprocal Rank)** = 1 / rang prvog relevantnog dokumenta medju vracenim (0 ako nijedan nije relevantan); **MRR** je prosek RR preko svih pitanja

In [11]:
# Pomocna celija - NEZAVISAN pregled cele baze (ne oslanja se na retriever).
# Ispisuje SVE jedinstvene izvor_fajl vrednosti koje postoje u vector_db, sortirano,
# da mozes tacno da prepoznas i kopiras naziv fajla za svaki lek (case-sensitive!).

svi_izvori = set()
for doc_id in vector_db.docstore._dict:
    doc = vector_db.docstore._dict[doc_id]
    svi_izvori.add(doc.metadata.get("izvor_fajl"))

for izvor in sorted(svi_izvori, key=lambda x: (x is None, x)):
    print(izvor)

print(f"\nUkupno jedinstvenih izvor_fajl vrednosti: {len(svi_izvori)}")

000019563 2024 59010 007 000 515 002 04 001.pdf
000020576 2024 59010 007 000 515 002 04 001.pdf
000456529 2023 59010 007 000 515 021 04 001.pdf
000456975 2023 59010 007 000 515 021 04 001.pdf
000457297 2023 59010 007 000 515 021 04 001.pdf
000457346 2023 59010 007 000 515 021 04 001.pdf
000457360 2023 59010 007 000 515 021 04 001.pdf
000457379 2023 59010 007 000 515 021 04 001.pdf
000457553 2023 59010 007 000 515 021 04 001.pdf
000457557 2023 59010 007 000 515 021 04 001.pdf
000457647 2023 59010 007 000 515 021 04 001.pdf
000457704 2023 59010 007 000 515 021 04 001.pdf
000457796 2023 59010 007 000 515 021 04 001.pdf
000457833 2023 59010 007 000 515 021 04 001.pdf
000461289 2023 59010 007 000 515 020 04 001.pdf
001977657 2024 59010 007 000 515 021 04 001.pdf
002132821 2024 59010 007 000 515 021 04 001.pdf
002641926 2024 59010 007 000 515 021 04 001.pdf
003353264 2024 59010 007 000 515 021 04 002.pdf
003503723 2024 59010 007 000 515 021 04 001.pdf
4.pdf
515-01-00007-22-002.pdf
515-01-000

In [24]:
# Pomocna celija - ispisi koje izvor_fajl vrednosti retriever vraca za dato pitanje.
# Iskoristi ovo da popunis "relevantni_izvori" u eval_dataset_retrieval ispod
# (kopiraj tacne nazive fajlova iz svoje baze).

primer_docs = retriever.invoke("Da li deca smeju da konzumiraju Pressing sirup?")
for d in primer_docs:
    print(d.metadata.get("izvor_fajl"), "\n", d.page_content[:200], "\n---")

000456529 2023 59010 007 000 515 021 04 001.pdf 
 Lek Pressing sirup ne smete uzimati:
ukoliko imate preosetljivi (alergijčni) na loratadin ili na bilo koju od pomoćnih supstanci ovog leka ( _videti odeljak 6.“Sadržaj pomoćnih supstanci”_ ). 
---
000456529 2023 59010 007 000 515 021 04 001.pdf 
 Upozorenja i mere opreza
Razgovarajte sa Vašim lekarom ili farmaceutom pre nego što uzmete lek Pressing sirup:  
- Ukoliko imate teško oboljenje jetre  
- Ukoliko treba da uradite bilo kakve kožne ale 
---
000456529 2023 59010 007 000 515 021 04 001.pdf 
 Uzimanje leka Pressing sirup sa hranom, pićima i alkoholom
Lek Pressing sirup se može uzimati bez obzira na vreme obroka. Lek Pressing sirup, ne pojačava efekte alkoholnih pića. 
---
000457297 2023 59010 007 000 515 021 04 001.pdf 
 Deca
Lek nije namenjen deci uzrasta ispod 12 godina. Za mlađu decu dostupne su pogodnije formulacije u obliku sirupa. 
---
000456529 2023 59010 007 000 515 021 04 001.pdf 
 Lek Pressing sirup sadrži saharozu i prop

In [13]:
# TODO: popuni "relevantni_izvori" tacnim vrednostima izvor_fajl iz tvoje baze
# (jedno pitanje moze imati jedan ili vise relevantnih fajlova/chunkova).
# Pitanja su preuzeta iz test primera iz glavnog notebook-a.

eval_dataset_retrieval = [
    {
        "pitanje": "Kada ne smem da uzimam lek Dovicin?",
        "relevantni_izvori": ["515-01-00434-19-001.pdf"],
    },
    {
        "pitanje": "Koja je uobičajena doza za lek Nimulid?",
        "relevantni_izvori": ["515-01-05090-22-001.pdf"],
    },
    {
        "pitanje": "Da li deca smeju da konzumiraju Pressing sirup?",
        "relevantni_izvori": ["000456529 2023 59010 007 000 515 021 04 001.pdf"],
    },
    {
        "pitanje": "Za šta se primenjuje lek Zoloft",
        "relevantni_izvori": ["515-01-00535-22-001.pdf"],
    },

]

len(eval_dataset_retrieval)

4

In [25]:
def precision_at_k(retrieved_ids, relevant_ids, k):
    if k == 0:
        return 0.0
    top_k = retrieved_ids[:k]
    pogodaka = sum(1 for r in top_k if r in relevant_ids)
    return pogodaka / k


def recall_at_k(retrieved_ids, relevant_ids, k):
    if len(relevant_ids) == 0:
        return 0.0
    top_k = set(retrieved_ids[:k])
    pogodaka = sum(1 for r in top_k if r in relevant_ids)
    return pogodaka / len(relevant_ids)


def hit_rate_at_k(retrieved_ids, relevant_ids, k):
    top_k = retrieved_ids[:k]
    return 1.0 if any(r in relevant_ids for r in top_k) else 0.0


def reciprocal_rank(retrieved_ids, relevant_ids):
    for rang, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            return 1.0 / rang
    return 0.0

In [26]:
rezultati_retrieval = []

for primer in eval_dataset_retrieval:
    pitanje = primer["pitanje"]
    relevantni = set(primer["relevantni_izvori"])

    povuceni_docs = retriever.invoke(pitanje)
    povuceni_ids = [d.metadata.get("izvor_fajl") for d in povuceni_docs]

    p = precision_at_k(povuceni_ids, relevantni, K)
    r = recall_at_k(povuceni_ids, relevantni, K)
    hr = hit_rate_at_k(povuceni_ids, relevantni, K)
    rr = reciprocal_rank(povuceni_ids, relevantni)

    rezultati_retrieval.append({
        "Pitanje": pitanje,
        f"Precision@{K}": round(p, 3),
        f"Recall@{K}": round(r, 3),
        f"Hit Rate@{K}": round(hr, 3),
        "RR (MRR baza)": round(rr, 3),
    })

df_retrieval = pd.DataFrame(rezultati_retrieval)
df_retrieval

,Pitanje,Precision@5,Recall@5,Hit Rate@5,RR (MRR baza)
0,Kada ne smem da uzimam lek Dovicin?,0.8,1.0,1.0,0.5
1,Koja je uobicajena doza za lek Nimulid?,1.0,1.0,1.0,1.0
2,Da li deca smeju da konzumiraju Pressing sirup?,0.8,1.0,1.0,1.0
3,Za sta se primenjuje lek Zoloft,1.0,1.0,1.0,1.0


In [27]:
# Konacna, agregirana tabela RAG (retrieval) evaluacije
prosek_red = {
    "Pitanje": "PROSEK (sva pitanja)",
    f"Precision@{K}": round(df_retrieval[f"Precision@{K}"].mean(), 3),
    f"Recall@{K}": round(df_retrieval[f"Recall@{K}"].mean(), 3),
    f"Hit Rate@{K}": round(df_retrieval[f"Hit Rate@{K}"].mean(), 3),
    "RR (MRR baza)": round(df_retrieval["RR (MRR baza)"].mean(), 3),
}

konacna_tabela_retrieval = pd.concat(
    [df_retrieval, pd.DataFrame([prosek_red])],
    ignore_index=True
)
# poslednji red = MRR (prosek RR kolone) i prosek ostalih metrika

print("=" * 100)
print("KONACNA TABELA - EVALUACIJA RAG (RETRIEVAL) KOMPONENTE")
print("=" * 100)
print(konacna_tabela_retrieval.to_string(index=False))
konacna_tabela_retrieval

KONACNA TABELA - EVALUACIJA RAG (RETRIEVAL) KOMPONENTE
                                        Pitanje  Precision@5  Recall@5  Hit Rate@5  RR (MRR baza)
            Kada ne smem da uzimam lek Dovicin?          0.8       1.0         1.0          0.500
        Koja je uobicajena doza za lek Nimulid?          1.0       1.0         1.0          1.000
Da li deca smeju da konzumiraju Pressing sirup?          0.8       1.0         1.0          1.000
                Za sta se primenjuje lek Zoloft          1.0       1.0         1.0          1.000
                           PROSEK (sva pitanja)          0.9       1.0         1.0          0.875


,Pitanje,Precision@5,Recall@5,Hit Rate@5,RR (MRR baza)
0,Kada ne smem da uzimam lek Dovicin?,0.8,1.0,1.0,0.500
1,Koja je uobicajena doza za lek Nimulid?,1.0,1.0,1.0,1.000
2,Da li deca smeju da konzumiraju Pressing sirup?,0.8,1.0,1.0,1.000
3,Za sta se primenjuje lek Zoloft,1.0,1.0,1.0,1.000
4,PROSEK (sva pitanja),0.9,1.0,1.0,0.875


In [28]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=konacna_tabela_retrieval)

https://docs.google.com/spreadsheets/d/1erI4-dmBj2qv3Fal4A4jSbgGw7c9P2L3FRs4AggybAQ/edit#gid=0


Recall se ispostavio kao nepotrebna metrika!

---
# Deo 2: Evaluacija generisanih odgovora

Ovde se evaluira **finalni tekstualni odgovor** modela (posle RAG-a) u odnosu na referentni (ocekivani, rucno napisan) odgovor. Sve metrike su besplatne/lokalne - bez LLM-as-judge.

Formule / definicije:

- **ROUGE-L (F1)** - zasnovan na najduzem zajednickom podnizu (LCS) izmedju generisanog i referentnog odgovora:
  - `R_lcs = LCS(gen, ref) / len(ref)`
  - `P_lcs = LCS(gen, ref) / len(gen)`
  - `ROUGE-L = (1 + beta^2) * P_lcs * R_lcs / (R_lcs + beta^2 * P_lcs)`
- **BLEU** - preciznost preklapanja n-grama (n=1..4) izmedju generisanog i referentnog teksta, sa "brevity penalty" (BP) kaznom za prekratke odgovore:
  - `BLEU = BP * exp( sum_{n=1..4} (1/4) * log(p_n) )`
- **BERTScore (F1)** - kosinusna slicnost kontekstualnih BERT embedinga izmedju tokena generisanog i referentnog odgovora (koristi multijezicki `bert-base-multilingual-cased`, pogodan za srpski):
  - `P = (1/|gen|) * sum_{x u gen} max_{y u ref} cos(x, y)`
  - `R = (1/|ref|) * sum_{y u ref} max_{x u gen} cos(x, y)`
  - `F1 = 2 * P * R / (P + R)`
- **Cosine similarity (embedding nivo)** - kosinusna slicnost celih recenica, koriscenjem istog embedding modela kao i retriever (`paraphrase-multilingual-mpnet-base-v2`):
  - `cos_sim = (E(gen) . E(ref)) / (||E(gen)|| * ||E(ref)||)`

In [17]:
# TODO: popuni "referentni_odgovor" - ocekivani/tacan odgovor na osnovu tvoje baze znanja
# (kratak, cinjenicno tacan odgovor koji bi idealan model trebalo da da).

eval_dataset_odgovori = [
    {
        "pitanje": "Kada ne smem da uzimam lek Dovicin?",
        "referentni_odgovor": '''Lek Dovicin ne smete uzimati:
- ukoliko ste alergični (preosetljivi) na doksiciklin ili neki drugi lek iz grupe tetraciklina ili na bilo koju od od pomoćnih supstanci ovog leka (navedene u odeljku 6);
- ukoliko ste trudni ili planirate trudnoću;
- ukoliko dojite;
- ukoliko imate oboljenje praćeno otežanim gutanjem.
Lek Dovicin ne treba koristiti tokom perioda razvoja zuba (u periodu trudnoće, kod odojčadi i dece mlađe od 8 godina)
''',
    },
    {
        "pitanje": "Koja je uobičajena doza za lek Nimulid?",
        "referentni_odgovor": "Lek Nimulid uvek uzimajte tačno onako kako Vam je objasnio Vaš lekar. Ukoliko niste sigurni, proverite sa Vašim lekarom ili farmaceutom. Uobičajena doza leka je jedna tableta (100 mg) dva puta dnevno, posle obroka.",
    },
    {
        "pitanje": "Da li deca smeju da konzumiraju Pressing?",
        "referentni_odgovor": "Deca: Nemojre davati lel Pressing sirup deci mlađoj od 2 godine.",
    },
    {
        "pitanje": "Za šta se primenjuje lek Zoloft",
        "referentni_odgovor": '''Lek Zoloft se može primenjivati za lečenje:
• Depresije i prevenciju ponovnog javljanja depresije (kod odraslih).
• Socijalnog anksioznog poremećaja (kod odraslih).
• Posttraumatskog stresnog poremećaja (kod odraslih).
• Paničnog poremećaja (kod odraslih).
• Opsesivno-kompulzivnog poremećaja (kod odraslih i dece i adolescenata uzrasta od 6 do 17 godina).''',
    },

]

len(eval_dataset_odgovori)

4

In [18]:
# Generisanje odgovora modela za svako test-pitanje (izolovano, bez istorije)

generisani_odgovori = []
for primer in eval_dataset_odgovori:
    odgovor, _ = generisi_odgovor(primer["pitanje"])
    generisani_odgovori.append(odgovor)

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentati

Q: Kada ne smem da uzimam lek Dovicin?
A: 

Ne smete uzimat' lek Dovičin ukoliko:

- ste alergičan (preosvetljiv) na dokičiklin ili nekli drugi leik iz grupe tetraciklina, ili na bili koju od pomoći supstancija ovog leika (navedena u odelju 6);
- ste trudne ili planirate roditeljstvo;
- dojite;
- imate obolenje praćenim otežnim gutanjem.

Pre upotrebu svakog leka obavezno saovetujte se sa lekarim ili farmaceutom.
[41.5s]



[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Koja je uobicajena doza za lek Nimulid?
A: 

Uobičajan doza lek Nimuld je jedna tablete (100 mg), dva puta dana, posle jela.
[7.3s]



[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Da li deca smeju da konzumiraju Pressing?
A: 

Nažalosti, nemam informaciju o dozama i upotrebi leka Pressing kod dece. Najbolje je da se savetujete sa lekarjem ili farmaceutima pre upotrebe bilo kog leka, posebno ako je pacijenat deca.
[15.7s]

Q: Za sta se primenjuje lek Zoloft
A: 

Lek Zolof se može koristiti za lečenju sledećih stanja:

- Depresija i prevencija ponovnoga javljanje depresija kod odrasle populacije.
- Socijalni poremeće anksiozanog tipa kod odrasnih.
- Posttraumatiski stresni sindrom kod odraslaca.
- Paničan poremećai kod odrastih.
- Opsesio-kompulzijski poremeća kod održavajućih pacijenata od 6 godina i više.
[26.2s]



In [19]:
rouge_l_scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
smoothing = SmoothingFunction().method1


def racunaj_rouge_l(generisan, referentan):
    rezultat = rouge_l_scorer.score(referentan, generisan)
    return rezultat["rougeL"].fmeasure


def racunaj_bleu(generisan, referentan):
    referenca_tok = [referentan.split()]
    kandidat_tok = generisan.split()
    return sentence_bleu(referenca_tok, kandidat_tok, smoothing_function=smoothing)


def racunaj_cos_similarity(generisan, referentan):
    emb_gen = embedding_model.embed_query(generisan)
    emb_ref = embedding_model.embed_query(referentan)
    sim = cosine_similarity([emb_gen], [emb_ref])[0][0]
    return float(sim)

In [20]:
referentni_odgovori = [p["referentni_odgovor"] for p in eval_dataset_odgovori]

# BERTScore se racuna u batch-u (brze i preciznije nego jedan po jedan)
bert_p, bert_r, bert_f1 = bert_score_fn(
    generisani_odgovori,
    referentni_odgovori,
    lang="sr",
    verbose=False,
)

rezultati_odgovora = []
for i, primer in enumerate(eval_dataset_odgovori):
    gen = generisani_odgovori[i]
    ref = primer["referentni_odgovor"]

    rezultati_odgovora.append({
        "Pitanje": primer["pitanje"],
        "ROUGE-L": round(racunaj_rouge_l(gen, ref), 3),
        "BLEU": round(racunaj_bleu(gen, ref), 3),
        "BERTScore (F1)": round(bert_f1[i].item(), 3),
        "Cosine similarity": round(racunaj_cos_similarity(gen, ref), 3),
    })

df_odgovori = pd.DataFrame(rezultati_odgovora)
df_odgovori

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,Pitanje,ROUGE-L,BLEU,BERTScore (F1),Cosine similarity
0,Kada ne smem da uzimam lek Dovicin?,0.406,0.016,0.819,0.945
1,Koja je uobicajena doza za lek Nimulid?,0.346,0.012,0.793,0.848
2,Da li deca smeju da konzumiraju Pressing?,0.047,0.006,0.636,0.654
3,Za sta se primenjuje lek Zoloft,0.362,0.014,0.801,0.964


In [21]:
# Konacna, agregirana tabela evaluacije odgovora
prosek_red_odg = {
    "Pitanje": "PROSEK (sva pitanja)",
    "ROUGE-L": round(df_odgovori["ROUGE-L"].mean(), 3),
    "BLEU": round(df_odgovori["BLEU"].mean(), 3),
    "BERTScore (F1)": round(df_odgovori["BERTScore (F1)"].mean(), 3),
    "Cosine similarity": round(df_odgovori["Cosine similarity"].mean(), 3),
}

konacna_tabela_odgovori = pd.concat(
    [df_odgovori, pd.DataFrame([prosek_red_odg])],
    ignore_index=True
)

print("=" * 100)
print("KONACNA TABELA - EVALUACIJA GENERISANIH ODGOVORA")
print("=" * 100)
print(konacna_tabela_odgovori.to_string(index=False))
konacna_tabela_odgovori

KONACNA TABELA - EVALUACIJA GENERISANIH ODGOVORA
                                  Pitanje  ROUGE-L  BLEU  BERTScore (F1)  Cosine similarity
      Kada ne smem da uzimam lek Dovicin?    0.406 0.016           0.819              0.945
  Koja je uobicajena doza za lek Nimulid?    0.346 0.012           0.793              0.848
Da li deca smeju da konzumiraju Pressing?    0.047 0.006           0.636              0.654
          Za sta se primenjuje lek Zoloft    0.362 0.014           0.801              0.964
                     PROSEK (sva pitanja)    0.290 0.012           0.762              0.853


,Pitanje,ROUGE-L,BLEU,BERTScore (F1),Cosine similarity
0,Kada ne smem da uzimam lek Dovicin?,0.406,0.016,0.819,0.945
1,Koja je uobicajena doza za lek Nimulid?,0.346,0.012,0.793,0.848
2,Da li deca smeju da konzumiraju Pressing?,0.047,0.006,0.636,0.654
3,Za sta se primenjuje lek Zoloft,0.362,0.014,0.801,0.964
4,PROSEK (sva pitanja),0.290,0.012,0.762,0.853


In [23]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=konacna_tabela_odgovori)

https://docs.google.com/spreadsheets/d/1PnIijRhMRN3VJHGKFqOlEXAWxdnfPtzUOH8wI0KuwmA/edit#gid=0
